# Problema 21: Integracion Temporal con Metodo de Newmark

## Relacion 5 - Analisis Time-History

**Objetivo:** Implementar el metodo de Newmark para la integracion temporal de la ecuacion de movimiento de un sistema SDOF sometido a excitacion sismica.

### Enunciado

Se tiene un sistema de un grado de libertad con:
- Masa: $m = 1000$ kg
- Rigidez: $k = 40000$ N/m
- Amortiguamiento: $\zeta = 5\%$

El sistema esta sometido a un pulso sismico sinusoidal en la base:
$$\ddot{u}_g(t) = A_0 \sin(\omega_{exc} t) \quad \text{para } t \le t_p$$

Con:
- Amplitud: $A_0 = 2.0$ m/s$^2$
- Frecuencia: $\omega_{exc} = 10$ rad/s
- Duracion: $t_p = \pi/\omega_{exc}$ s (medio ciclo)

Se pide resolver la respuesta dinamica mediante el metodo de Newmark.

## Datos del Problema

**Sistema estructural:**
| Parametro | Valor | Unidad |
|-----------|-------|--------|
| $m$ | 1000 | kg |
| $k$ | 40000 | N/m |
| $\zeta$ | 5 | % |

**Excitacion:**
| Parametro | Valor | Unidad |
|-----------|-------|--------|
| $A_0$ | 2.0 | m/s$^2$ |
| $\omega_{exc}$ | 10 | rad/s |

In [ ]:
# Importacion de librerias
import numpy as np
import matplotlib.pyplot as plt

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

# Parametros del sistema
m = 1000    # kg - masa
k = 40000   # N/m - rigidez
zeta = 0.05 # amortiguamiento

# Propiedades dinamicas
omega_n = np.sqrt(k/m)       # frecuencia natural (rad/s)
T_n = 2*np.pi/omega_n        # periodo natural (s)
c = 2 * zeta * m * omega_n   # coeficiente de amortiguamiento
omega_d = omega_n * np.sqrt(1 - zeta**2)  # frecuencia amortiguada

print("PROPIEDADES DEL SISTEMA")
print("="*40)
print(f"Masa: m = {m} kg")
print(f"Rigidez: k = {k} N/m")
print(f"Amortiguamiento: zeta = {zeta*100:.0f}%")
print(f"\nPropiedades dinamicas:")
print(f"  omega_n = {omega_n:.2f} rad/s")
print(f"  f_n = {omega_n/(2*np.pi):.2f} Hz")
print(f"  T_n = {T_n:.3f} s")
print(f"  c = {c:.1f} Ns/m")

In [ ]:
# Parametros de la excitacion
A0 = 2.0       # m/s^2 - amplitud del pulso
omega_exc = 10  # rad/s - frecuencia de excitacion
t_p = np.pi / omega_exc  # duracion del pulso

# Ratio de frecuencias
r = omega_exc / omega_n

print("PARAMETROS DE LA EXCITACION")
print("="*40)
print(f"Amplitud: A0 = {A0} m/s^2")
print(f"Frecuencia: omega_exc = {omega_exc} rad/s")
print(f"Duracion del pulso: t_p = {t_p:.4f} s")
print(f"\nRatio de frecuencias: r = omega_exc/omega_n = {r:.2f}")

## 1. Definicion de la Excitacion

Pulso sismico sinusoidal de medio ciclo:

In [ ]:
def ag(t, A0, omega_exc, t_p):
    """
    Aceleracion en la base (pulso sinusoidal).
    
    Parametros:
        t: tiempo (puede ser array)
        A0: amplitud
        omega_exc: frecuencia de excitacion
        t_p: duracion del pulso
    
    Retorna:
        aceleracion en la base
    """
    if isinstance(t, np.ndarray):
        return np.where(t <= t_p, A0 * np.sin(omega_exc * t), 0.0)
    return A0 * np.sin(omega_exc * t) if t <= t_p else 0.0

# Visualizar la excitacion
t_plot = np.linspace(0, 1.5, 500)
ag_plot = ag(t_plot, A0, omega_exc, t_p)

plt.figure(figsize=(10, 4))
plt.plot(t_plot, ag_plot, 'r-', linewidth=2)
plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
plt.axvline(x=t_p, color='gray', linestyle='--', alpha=0.7, label=f't_p = {t_p:.3f} s')
plt.fill_between(t_plot, ag_plot, alpha=0.3, color='red')
plt.xlabel('Tiempo (s)')
plt.ylabel('Aceleracion en la base (m/s^2)')
plt.title('Excitacion sismica (pulso sinusoidal)', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 1.5)
plt.show()

## 2. Metodo de Newmark

### Ecuacion de movimiento

Para excitacion sismica en la base:
$$m\ddot{u} + c\dot{u} + ku = -m\ddot{u}_g(t)$$

Donde $u$ es el desplazamiento relativo respecto a la base.

### Algoritmo de Newmark

El metodo de Newmark asume:
$$\dot{u}_{i+1} = \dot{u}_i + [(1-\gamma)\ddot{u}_i + \gamma\ddot{u}_{i+1}]\Delta t$$
$$u_{i+1} = u_i + \dot{u}_i \Delta t + [(\frac{1}{2}-\beta)\ddot{u}_i + \beta\ddot{u}_{i+1}]\Delta t^2$$

Para el metodo de **aceleracion media constante** (estable incondicionalmente):
- $\gamma = 1/2$
- $\beta = 1/4$

In [ ]:
def newmark_integration(m, c, k, p, dt, gamma=0.5, beta=0.25, u0=0, v0=0):
    """
    Integracion temporal mediante el metodo de Newmark.
    
    Parametros:
        m: masa
        c: amortiguamiento
        k: rigidez
        p: vector de fuerzas (array)
        dt: paso de tiempo
        gamma, beta: parametros de Newmark
        u0, v0: condiciones iniciales
    
    Retorna:
        u, v, a: arrays de desplazamiento, velocidad y aceleracion
    """
    n = len(p)
    
    # Coeficientes de Newmark
    a0 = 1 / (beta * dt**2)
    a1 = gamma / (beta * dt)
    a2 = 1 / (beta * dt)
    a3 = 1 / (2*beta) - 1
    a4 = gamma / beta - 1
    a5 = dt * (gamma / (2*beta) - 1)
    
    # Rigidez efectiva
    k_eff = k + a0*m + a1*c
    
    # Arrays de respuesta
    u = np.zeros(n)
    v = np.zeros(n)
    a = np.zeros(n)
    
    # Condiciones iniciales
    u[0] = u0
    v[0] = v0
    a[0] = (p[0] - c*v0 - k*u0) / m
    
    # Integracion paso a paso
    for i in range(n-1):
        # Carga efectiva
        p_eff = p[i+1] + m*(a0*u[i] + a2*v[i] + a3*a[i]) + c*(a1*u[i] + a4*v[i] + a5*a[i])
        
        # Resolver
        u[i+1] = p_eff / k_eff
        a[i+1] = a0*(u[i+1] - u[i]) - a2*v[i] - a3*a[i]
        v[i+1] = v[i] + dt*((1-gamma)*a[i] + gamma*a[i+1])
    
    return u, v, a

print("FUNCION newmark_integration DEFINIDA")
print("="*40)
print("Parametros por defecto (aceleracion media):")
print("  gamma = 0.5")
print("  beta = 0.25")

## 3. Integracion de la Respuesta

In [ ]:
# Parametros de integracion
dt = 0.005   # paso de tiempo (s)
t_max = 1.5  # tiempo total (s)

# Vector de tiempo
t = np.arange(0, t_max + dt, dt)
n_steps = len(t)

# Vector de aceleracion en la base
ag_hist = ag(t, A0, omega_exc, t_p)

# Vector de fuerza efectiva (F = -m * ag)
p = -m * ag_hist

# Integrar usando Newmark
u, v, a = newmark_integration(m, c, k, p, dt)

# Aceleracion absoluta
a_abs = a + ag_hist

print("PARAMETROS DE INTEGRACION")
print("="*40)
print(f"Paso de tiempo: dt = {dt*1000:.1f} ms")
print(f"Tiempo total: t_max = {t_max} s")
print(f"Numero de pasos: n = {n_steps}")
print(f"Ratio dt/T_n = {dt/T_n:.4f} (debe ser < 0.1 para precision)")

## 4. Resultados

In [ ]:
# Valores maximos
u_max = np.max(np.abs(u))
v_max = np.max(np.abs(v))
a_abs_max = np.max(np.abs(a_abs))

# Instantes de maximos
t_u_max = t[np.argmax(np.abs(u))]
t_v_max = t[np.argmax(np.abs(v))]
t_a_max = t[np.argmax(np.abs(a_abs))]

print("RESPUESTAS MAXIMAS")
print("="*50)
print(f"Desplazamiento maximo: u_max = {u_max*1000:.2f} mm en t = {t_u_max:.3f} s")
print(f"Velocidad maxima: v_max = {v_max*1000:.2f} mm/s en t = {t_v_max:.3f} s")
print(f"Aceleracion absoluta maxima: a_abs_max = {a_abs_max:.2f} m/s^2 en t = {t_a_max:.3f} s")

# Factor de amplificacion dinamica
u_estatico = m * A0 / k  # desplazamiento estatico equivalente
DAF = u_max / u_estatico
print(f"\nFactor de amplificacion dinamica (DAF): {DAF:.2f}")

## 5. Visualizacion de Resultados

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Excitacion
ax1 = axes[0, 0]
ax1.plot(t, ag_hist, 'r-', linewidth=2)
ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax1.axvline(x=t_p, color='gray', linestyle='--', alpha=0.7, label=f't_p = {t_p:.3f} s')
ax1.fill_between(t, ag_hist, alpha=0.3, color='red')
ax1.set_xlabel('Tiempo (s)')
ax1.set_ylabel('Aceleracion base (m/s^2)')
ax1.set_title('Excitacion sismica (pulso sinusoidal)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, t_max)

# Panel 2: Desplazamiento
ax2 = axes[0, 1]
ax2.plot(t, u*1000, 'b-', linewidth=1.5)
ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
i_max = np.argmax(np.abs(u))
ax2.plot(t[i_max], u[i_max]*1000, 'ro', markersize=10, 
         label=f'u_max = {u[i_max]*1000:.1f} mm')
ax2.set_xlabel('Tiempo (s)')
ax2.set_ylabel('Desplazamiento relativo (mm)')
ax2.set_title('Respuesta en desplazamiento', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, t_max)

# Panel 3: Velocidad
ax3 = axes[1, 0]
ax3.plot(t, v*1000, 'g-', linewidth=1.5)
ax3.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax3.set_xlabel('Tiempo (s)')
ax3.set_ylabel('Velocidad relativa (mm/s)')
ax3.set_title('Respuesta en velocidad', fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, t_max)

# Panel 4: Aceleracion absoluta
ax4 = axes[1, 1]
ax4.plot(t, a_abs, 'm-', linewidth=1.5, label='Aceleracion absoluta')
ax4.plot(t, ag_hist, 'r--', linewidth=1, alpha=0.5, label='Excitacion')
ax4.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax4.set_xlabel('Tiempo (s)')
ax4.set_ylabel('Aceleracion (m/s^2)')
ax4.set_title('Aceleracion absoluta', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xlim(0, t_max)

plt.tight_layout()
plt.show()

## 6. Diagrama de Fase y Energia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Diagrama de fase
ax1 = axes[0]
ax1.plot(u*1000, v*1000, 'b-', linewidth=1)
ax1.plot(u[0]*1000, v[0]*1000, 'go', markersize=10, label='Inicio')
ax1.plot(u[-1]*1000, v[-1]*1000, 'rs', markersize=10, label='Final')
ax1.set_xlabel('Desplazamiento (mm)')
ax1.set_ylabel('Velocidad (mm/s)')
ax1.set_title('Diagrama de fase', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# Panel 2: Energia
ax2 = axes[1]

# Calcular energias
E_cin = 0.5 * m * v**2           # Energia cinetica
E_pot = 0.5 * k * u**2           # Energia potencial elastica
E_total = E_cin + E_pot          # Energia total (sin disipacion)

ax2.plot(t, E_cin, 'b-', linewidth=1.5, label='Energia cinetica')
ax2.plot(t, E_pot, 'r-', linewidth=1.5, label='Energia potencial')
ax2.plot(t, E_total, 'k--', linewidth=1.5, label='Energia total (E_c + E_p)')

ax2.set_xlabel('Tiempo (s)')
ax2.set_ylabel('Energia (J)')
ax2.set_title('Balance energetico', fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, t_max)

plt.tight_layout()
plt.show()

print(f"Energia maxima: E_max = {np.max(E_total):.2f} J")
print(f"Energia final: E_final = {E_total[-1]:.2f} J")
print(f"Energia disipada: E_dis = {np.max(E_total) - E_total[-1]:.2f} J")

## 7. Estudio de Convergencia

In [ ]:
# Estudio de convergencia con diferentes pasos de tiempo
dt_values = [0.02, 0.01, 0.005, 0.002, 0.001]
u_max_values = []

print("ESTUDIO DE CONVERGENCIA")
print("="*50)
print(f"{'dt (ms)':<12} {'n_pasos':<12} {'u_max (mm)':<15} {'Error (%)':<12}")
print("-"*50)

for dt_test in dt_values:
    t_test = np.arange(0, t_max + dt_test, dt_test)
    ag_test = ag(t_test, A0, omega_exc, t_p)
    p_test = -m * ag_test
    u_test, _, _ = newmark_integration(m, c, k, p_test, dt_test)
    u_max_test = np.max(np.abs(u_test))
    u_max_values.append(u_max_test)

# Usar el valor con dt mas pequeno como referencia
u_ref = u_max_values[-1]

for dt_test, u_max_test in zip(dt_values, u_max_values):
    n_pasos = int(t_max / dt_test)
    error = abs(u_max_test - u_ref) / u_ref * 100
    print(f"{dt_test*1000:<12.1f} {n_pasos:<12} {u_max_test*1000:<15.3f} {error:<12.3f}")

print(f"\nValor de referencia (dt=1ms): u_max = {u_ref*1000:.3f} mm")

## Interpretacion de Resultados

### Respuesta del sistema
- Durante la fase de excitacion ($t < t_p$), el sistema se excita y acumula energia.
- Despues del pulso ($t > t_p$), el sistema vibra libremente con amortiguamiento.
- La respuesta maxima ocurre poco despues del fin del pulso.

### Factor de amplificacion
- El ratio de frecuencias $r = \omega_{exc}/\omega_n$ determina la amplificacion.
- Para $r \approx 1$ (resonancia), la amplificacion seria maxima.
- En este caso $r \approx 1.58$, por lo que hay amplificacion moderada.

### Metodo de Newmark
- El metodo de aceleracion media ($\gamma=0.5$, $\beta=0.25$) es incondicionalmente estable.
- No hay introduccion de amortiguamiento numerico.
- El paso de tiempo debe ser suficientemente pequeno para capturar la dinamica (tipicamente $\Delta t < T_n/10$).

### Balance energetico
- Durante la vibracion libre, la energia se disipa gradualmente por amortiguamiento.
- El diagrama de fase muestra la espiral convergente tipica de sistemas subamortiguados.